# Phase 3: SD-WAN Predictive Fault Analytics Engine
This notebook trains an air-gapped Random Forest model to detect network failure precursors and forecast outages before they impact OSPF/BGP routing states.

In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
import joblib
import warnings
warnings.filterwarnings('ignore')

### 1. Data Ingestion & Temporal Alignment
Merging the unsynchronized telemetry polling with the exact fault injection timestamps.

In [2]:
# Load datasets
telemetry_df = pd.read_csv('sdwan_telemetry.csv', parse_dates=['timestamp'])
labels_df = pd.read_csv('ml_ground_truth_labels.csv', parse_dates=['timestamp'])

# Sort both by timestamp for asof merging
telemetry_df = telemetry_df.sort_values('timestamp')
labels_df = labels_df.sort_values('timestamp')

# Merge datasets based on the closest timestamp (forward fill state)
df = pd.merge_asof(telemetry_df, labels_df, on='timestamp', direction='backward')
df['severity'] = df['severity'].fillna('normal')
df['fault_type'] = df['fault_type'].fillna('none')

### 2. Feature Engineering (Sliding Windows)
We calculate rolling means and standard deviations to capture network degradation trends over time.

In [3]:
window_size = 3 # Represents approx 10-15 seconds of polling
features = ['avg_latency_ms', 'jitter_ms', 'packet_loss_pct', 'rx_bytes_per_sec']

for feat in features:
    df[f'{feat}_rolling_mean'] = df[feat].rolling(window=window_size).mean()
    df[f'{feat}_rolling_std'] = df[feat].rolling(window=window_size).std()

# Drop initial NaN values caused by rolling windows
df = df.dropna()

### 3. Target Creation (Lead-Time Forecasting)
Instead of predicting current state, we shift the label backwards. 
Goal: If the network will fail in the next ~30 seconds, label the *current* precursor state as 1.

In [4]:
prediction_horizon_steps = 5 # Approx 15-25 seconds into the future

# Create binary target: 1 if Critical fault occurs, 0 otherwise
df['is_critical'] = (df['severity'] == 'critical').astype(int)

# Shift target backwards
df['target_future_failure'] = df['is_critical'].shift(-prediction_horizon_steps)
df = df.dropna()

# Feature Matrix (X) and Target (y)
feature_cols = [col for col in df.columns if 'rolling' in col]
X = df[feature_cols]
y = df['target_future_failure']

### 4. Temporal Train/Test Split & Training
We must not use standard random splits for time-series, or we leak future data into the past.

In [5]:
split_index = int(len(df) * 0.7)
X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]

# Train Random Forest
clf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
clf.fit(X_train, y_train)

print("Training Complete.")

Training Complete.


### 5. Evaluation & Serialization
Validating the early-warning capabilities.

In [6]:
y_pred = clf.predict(X_test)
print("Model Evaluation on Future Forecasting:\n")
print(classification_report(y_test, y_pred))

# Feature Importance for the LLM Copilot Context
importances = pd.DataFrame({'feature': feature_cols, 'importance': clf.feature_importances_})
print("\nTop Features driving failure predictions:")
print(importances.sort_values('importance', ascending=False).head(3))

# Save Model
joblib.dump(clf, 'predictive_brain.joblib')
print("\nModel successfully saved to predictive_brain.joblib")

Model Evaluation on Future Forecasting:

              precision    recall  f1-score   support

         0.0       0.99      0.87      0.93       150
         1.0       0.79      0.99      0.88        74

    accuracy                           0.91       224
   macro avg       0.89      0.93      0.90       224
weighted avg       0.93      0.91      0.91       224


Top Features driving failure predictions:
                       feature  importance
0  avg_latency_ms_rolling_mean    0.348742
1   avg_latency_ms_rolling_std    0.195522
2       jitter_ms_rolling_mean    0.176599

Model successfully saved to predictive_brain.joblib
